In [ ]:
import pickle 
import tifffile as tf 
import numpy as np 
import pandas as pd
from tqdm import tqdm
import cv2
from scipy.ndimage import label, binary_dilation
from skimage.measure import regionprops
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

In [ ]:
keys =  [] # list of sample identifiers for analysis

In [ ]:
PODXL_keys = ['Cycle 1_CH2', 'Cycle 1_CH2', 'Cycle 2_CH2']

In [ ]:
glomerulus_classification_path = r"" # path to glomerulosclerosis results

In [ ]:
registration_path = r"" # path to registered images

In [ ]:
cell_clustering_path = r"" # path to cell phenotyping results

In [ ]:
color_deconvolution_path = r"" # path to color deconvolution results

In [ ]:
cell_type_color_dict_paths = [cell_clustering_path + keys[k] + '/cell_type_color_dict.pickle' for k in range(len(keys))]

In [ ]:
cell_segmentation_mask_paths =  [cell_clustering_path + keys[k] + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle' for k in range(len(keys))]

In [ ]:
cell_cluster_location_df_paths = [cell_clustering_path + keys[k] + '/cell_cluster_location_merged_unclassified_within_contour.csv' for k in range(len(keys))]

In [ ]:
glomerulus_mask_paths = [glomerulus_classification_path + keys[k] + '/mask_registered.tif' for k in range(len(keys))]

In [ ]:
pas_images = [color_deconvolution_path + keys[k] + '/pas.tif' for k in range(len(keys))]

In [ ]:
PODXL_paths = []
i = 0
for k in keys: 
    registration_path_for_key = registration_path + '/' + k.split('/')[1] 
    PODXL_key = PODXL_keys[i]
    PODXL_path = [f for f in Path(registration_path_for_key).rglob('*.tif') if PODXL_key.split('_')[0] in str(f) and PODXL_key.split('_')[1] in str(f) and k.split('/')[2] in str(f)][0]
    PODXL_paths.append(PODXL_path)
    i+=1 

In [ ]:
podxl_image_dict = {}

for i in range(len(keys)): 
   podxl_image_dict[keys[i].split('/')[-1]] = tf.imread(PODXL_paths[i])

In [ ]:
trichrome_blue_type = '_non_wsi'

In [ ]:
trichrome_blue_images = [color_deconvolution_path + keys[k] + '/trichrome_blue' + trichrome_blue_type + '.tif' for k in range(len(keys))]

In [ ]:
cell_cluster_location_df_dict = {}
glomerulus_class_mask_dict = {}

for i in range(len(keys)): 
    cell_cluster_location_df_dict[keys[i].split('/')[-1]] = pd.read_csv(cell_cluster_location_df_paths[i]).drop(columns=['Unnamed: 0'])
    glomerulus_class_mask_dict[keys[i].split('/')[-1]] = tf.imread(glomerulus_mask_paths[i])

In [ ]:
trichrome_blue_image_dict = {}

for i in range(len(keys)): 

    trichrome_blue_image_dict[keys[i].split('/')[-1]] = tf.imread(trichrome_blue_images[i])

In [ ]:
pas_image_dict = {}
for i in range(len(keys)): 
    pas_image_dict[keys[i].split('/')[-1]] = tf.imread(pas_images[i])

In [ ]:
labels = {}
cell_segmentation_mask_dict = {}
for i in range(len(keys)):
    with open(glomerulus_classification_path + keys[i] + '/labels.pickle', 'rb') as f: 
        labels[keys[i].split('/')[-1]] = pickle.load(f)
    with open(cell_segmentation_mask_paths[i], 'rb') as f: 
        cell_segmentation_mask_dict[keys[i].split('/')[-1]] = pickle.load(f)

In [ ]:
cell_type_color_dict_map = {}
for i in range(len(keys)):
    with open(cell_type_color_dict_paths[i], 'rb') as f: 
        cell_type_color_dict_map[keys[i].split('/')[-1]] = pickle.load(f)

In [ ]:
glomerulus_labels = []
for sample_glomerulus_labels in labels.values():
    glomerulus_labels.extend(sample_glomerulus_labels)

In [ ]:
#for each glomerulus 
#calculate fibrosis outside the cell within a radius 
#calculate number of podocytes within cell

glomerulus_mask_dict = {}

glomerulus_mask_coordinates_dict = {}

for key, mask_class in glomerulus_class_mask_dict.items():
    mask_class_type_changed = (mask_class/ np.max(mask_class) * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask_class_type_changed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    mask_glomerulus = np.zeros((mask_class_type_changed.shape[0], mask_class_type_changed.shape[1]), dtype=np.uint8)

    coordinates  = []
    for i, contour in tqdm(enumerate(contours)):
        x, y, w, h = cv2.boundingRect(contour)
        coordinates.append((x, y, w, h))
        cv2.drawContours(
            mask_glomerulus,
            [contour],
            contourIdx=-1,
            color=i+1,     
            thickness=-1  
        )
            

    glomerulus_mask_dict[key] = mask_glomerulus
    glomerulus_mask_coordinates_dict[key] = coordinates


In [ ]:
r = 50

In [ ]:
sample_glomerulus_metrics = {}

In [ ]:

for key, coordinates in glomerulus_mask_coordinates_dict.items():

    mask_class = glomerulus_class_mask_dict[key]

    mask_glomerulus = glomerulus_mask_dict[key]

    cell_segmentation_mask = cell_segmentation_mask_dict[key]

    pas = pas_image_dict[key] 
    pas = 255 - pas 
    pas = pas / np.max(pas)

    trichrome_blue = trichrome_blue_image_dict[key]
    trichrome_blue = 255 - trichrome_blue 
    trichrome_blue = trichrome_blue / np.max(trichrome_blue)


    cell_cluster_location_df = cell_cluster_location_df_dict[key]

    glomerulus_metrics = []

    for i in tqdm(range(1, len(coordinates)+1)):

        x, y, w, h = coordinates[i-1]

        y_min = max(y - r, 0)
        y_max = min(y + h + r + 1, mask_class.shape[0])
        x_min = max(x - r, 0)
        x_max = min(x + w + r + 1, mask_class.shape[1])

        # Crop blocks
        glom_label_block = mask_glomerulus[y_min:y_max, x_min:x_max]
        glomerulus_block = glom_label_block == i

        cell_block = cell_segmentation_mask[y_min:y_max, x_min:x_max]
        fibrosis_block = trichrome_blue[y_min:y_max, x_min:x_max]
        pas_block = pas[y_min:y_max, x_min:x_max]

        # Expanded glomerulus mask
        expanded_glomerulus_mask = binary_dilation(
            glomerulus_block,
            structure=np.ones((r, r))
        )

        external_mask = expanded_glomerulus_mask & (~glomerulus_block)

        # -------------------
        # Cell metrics
        # -------------------
        glomerulus_cell_ids = np.unique(cell_block[glomerulus_block])
        glomerulus_cell_ids = glomerulus_cell_ids[glomerulus_cell_ids > 0]

        ys, xs = np.nonzero(cell_segmentation_mask)
        labels = cell_segmentation_mask[ys, xs]

        n = labels.max() + 1

        counts = np.bincount(labels, minlength=n)
        x_sum = np.bincount(labels, weights=xs, minlength=n)
        y_sum = np.bincount(labels, weights=ys, minlength=n)

        x_centroid = np.divide(x_sum, counts, where=counts > 0)
        y_centroid = np.divide(y_sum, counts, where=counts > 0)

        centroid_df = pd.DataFrame({
            "Cell ID": glomerulus_cell_ids,
            "X": x_centroid[glomerulus_cell_ids].round().astype(int),
            "Y": y_centroid[glomerulus_cell_ids].round().astype(int),
        })

        glomerulus_cells = cell_cluster_location_df.merge(
            centroid_df,
            on=["X", "Y"],
            how="inner"
        )

        podocyte_count = (glomerulus_cells["Cell Type"] == "Podocyte").sum()
        cell_count = len(glomerulus_cells)



        # -------------------
        # Fibrosis metrics
        # -------------------

        # Inside glomerulus
        inside_pixels = np.sum(glomerulus_block)

        if inside_pixels > 0:
            inside_fibrosis_values = fibrosis_block[glomerulus_block]

            inside_fibrosis_intensity_mean = inside_fibrosis_values.mean()

            inside_fibrosis_intensity_sum = inside_fibrosis_values.sum()

            inside_fibrosis_area_proportion = np.mean(inside_fibrosis_values > 0)

            inside_fibrosis_intensity_weighted_proportion = (
                inside_fibrosis_values.sum() / inside_pixels
            )
        else:
            inside_fibrosis_intensity_mean = 0.0
            inside_fibrosis_area_proportion = 0.0
            inside_fibrosis_intensity_weighted_proportion = 0.0
            inside_fibrosis_intensity_sum = 0.0

        # Outside glomerulus (within radius)
        outside_pixels = np.sum(external_mask)

        if outside_pixels > 0:
            outside_fibrosis_values = fibrosis_block[external_mask]

            outside_fibrosis_intensity_mean = outside_fibrosis_values.mean()

            outside_fibrosis_intensity_sum = outside_fibrosis_values.sum()

            outside_fibrosis_area_proportion = np.mean(outside_fibrosis_values > 0)

            outside_fibrosis_intensity_weighted_proportion = (
                outside_fibrosis_values.sum() / outside_pixels
            )
        else:
            outside_fibrosis_intensity_mean = 0.0
            outside_fibrosis_area_proportion = 0.0
            outside_fibrosis_intensity_weighted_proportion = 0.0
            outside_fibrosis_intensity_sum = 0.0

        #Inside and outside 

        all_pixels = np.sum(expanded_glomerulus_mask)

        if all_pixels > 0:
            all_fibrosis_values = fibrosis_block[expanded_glomerulus_mask]

            all_fibrosis_intensity_mean = all_fibrosis_values.mean()

            all_fibrosis_intensity_sum = all_fibrosis_values.sum()

            all_fibrosis_area_proportion = np.mean(all_fibrosis_values > 0)

            all_fibrosis_intensity_weighted_proportion = (
                all_fibrosis_values.sum() / all_pixels
            )
        else:
            all_fibrosis_intensity_mean = 0.0
            all_fibrosis_area_proportion = 0.0
            all_fibrosis_intensity_weighted_proportion = 0.0
            all_fibrosis_intensity_sum = 0.0



        # -------------------
        # PAS metrics 
        # -------------------

        # Inside glomerulus
        inside_pixels = np.sum(glomerulus_block)

        if inside_pixels > 0:
            inside_pas_values = pas_block[glomerulus_block]

            inside_pas_intensity_mean = inside_pas_values.mean()

            inside_pas_intensity_sum = inside_pas_values.sum()

            inside_pas_area_proportion = np.mean(inside_pas_values > 0)

            inside_pas_intensity_weighted_proportion = (
                inside_pas_values.sum() / inside_pixels
            )
        else:
            inside_pas_intensity_mean = 0.0
            inside_pas_area_proportion = 0.0
            inside_pas_intensity_weighted_proportion = 0.0
            inside_pas_intensity_sum = 0.0

        # Outside glomerulus (within radius)
        outside_pixels = np.sum(external_mask)

        if outside_pixels > 0:
            outside_pas_values = pas_block[external_mask]

            outside_pas_intensity_mean = outside_pas_values.mean()

            outside_pas_intensity_sum = outside_pas_values.sum()

            outside_pas_area_proportion = np.mean(outside_pas_values > 0)

            outside_pas_intensity_weighted_proportion = (
                outside_pas_values.sum() / outside_pixels
            )
        else:
            outside_pas_intensity_mean = 0.0
            outside_pas_area_proportion = 0.0
            outside_pas_intensity_weighted_proportion = 0.0
            outside_pas_intensity_sum = 0.0
        
        #Inside and outside 

        all_pixels = np.sum(expanded_glomerulus_mask)

        if all_pixels > 0:
            all_pas_values = pas_block[expanded_glomerulus_mask]

            all_pas_intensity_mean = all_pas_values.mean()

            all_pas_intensity_sum = all_pas_values.sum()

            all_pas_area_proportion = np.mean(all_pas_values > 0)

            all_pas_intensity_weighted_proportion = (
                all_pas_values.sum() / all_pixels
            )
        else:
            all_pas_intensity_mean = 0.0
            all_pas_area_proportion = 0.0
            all_pas_intensity_weighted_proportion = 0.0
            all_pas_intensity_sum = 0.0



        # -------------------
        # Collect metrics
        # -------------------
        glomerulus_metrics.append({
            "Podocyte Count": podocyte_count,
            "Cell Count": cell_count,

            "Internal Fibrosis Intensity Mean": inside_fibrosis_intensity_mean,
            "External Fibrosis Intensity Mean": outside_fibrosis_intensity_mean,
            "Total Fibrosis Intensity Mean": all_fibrosis_intensity_mean,

            "Internal Fibrosis Area Proportion": inside_fibrosis_area_proportion,
            "External Fibrosis Area Proportion": outside_fibrosis_area_proportion,
            "Total Fibrosis Area Proportion": all_fibrosis_area_proportion,

            "Internal Fibrosis Intensity-Weighted Proportion": inside_fibrosis_intensity_weighted_proportion,
            "External Fibrosis Intensity-Weighted Proportion": outside_fibrosis_intensity_weighted_proportion,
            "Total Fibrosis Intensity-Weighted Proportion": all_fibrosis_intensity_weighted_proportion,

            "Internal Fibrosis Intensity Sum": inside_fibrosis_intensity_sum,
            "External Fibrosis Intensity Sum": outside_fibrosis_intensity_sum,
            "Total Fibrosis Intensity Sum": all_fibrosis_intensity_sum,

            "Internal PAS Intensity Mean": inside_pas_intensity_mean,
            "External PAS Intensity Mean": outside_pas_intensity_mean,
            "Total PAS Intensity Mean": all_pas_intensity_mean,

            "Internal PAS Area Proportion": inside_pas_area_proportion,
            "External PAS Area Proportion": outside_pas_area_proportion,
            "Total PAS Area Proportion": all_pas_area_proportion,

            "Internal PAS Intensity-Weighted Proportion": inside_pas_intensity_weighted_proportion,
            "External PAS Intensity-Weighted Proportion": outside_pas_intensity_weighted_proportion,
            "Total PAS Intensity-Weighted Proportion": all_pas_intensity_weighted_proportion,

            "Internal PAS Intensity Sum": inside_pas_intensity_sum,
            "External PAS Intensity Sum": outside_pas_intensity_sum,
            "Total PAS Intensity Sum": all_pas_intensity_sum

            
        })

    sample_glomerulus_metrics[key] = pd.DataFrame(glomerulus_metrics)


In [ ]:
!pip install statannotations
from statannotations.Annotator import Annotator

In [ ]:
glomerulus_metrics_df = pd.concat(
    [
        df.assign(sample=key)
        for key, df in sample_glomerulus_metrics.items()
    ],
)

In [ ]:
glomerulus_metrics_df['Label'] = glomerulus_labels

In [ ]:
glomerulus_metrics_df['Podocyte Proportion'] = glomerulus_metrics_df['Podocyte Count']/glomerulus_metrics_df['Cell Count']

In [ ]:
glomerulus_metrics_df = glomerulus_metrics_df.reset_index()

In [ ]:
features = ["Podocyte Count", 'Podocyte Proportion', "Cell Count", 
            "Internal Fibrosis Intensity Mean","External Fibrosis Intensity Mean", "Internal Fibrosis Area Proportion", 
            "External Fibrosis Area Proportion", "Internal Fibrosis Intensity-Weighted Proportion", 
            "External Fibrosis Intensity-Weighted Proportion", "Internal Fibrosis Intensity Sum","External Fibrosis Intensity Sum", 
            "Internal PAS Intensity Mean", "External PAS Intensity Mean", "Internal PAS Area Proportion", "External PAS Area Proportion", 
            "Internal PAS Intensity-Weighted Proportion", "External PAS Intensity-Weighted Proportion", "Internal PAS Intensity Sum", 
            "External PAS Intensity Sum",
            "Total Fibrosis Intensity Mean", "Total Fibrosis Area Proportion", 
            "Total Fibrosis Intensity-Weighted Proportion", "Total Fibrosis Intensity Sum", 
            "Total PAS Intensity Mean", "Total PAS Area Proportion", 
            "Total PAS Intensity-Weighted Proportion", "Total PAS Intensity Sum"

            ]


In [ ]:
glomerulus_metrics_processed_df['Label'].unique()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf
import warnings

warnings.filterwarnings("ignore")

group = "Label"
sample_col = "sample"

colors = ["red", "dodgerblue"]

for feature in features:

    df = glomerulus_metrics_df[
        [sample_col, group, feature]
    ].dropna().copy()

    df[group] = pd.Categorical(
        df[group],
        categories=[1, 2]
    )

    model = smf.mixedlm(
        f'Q("{feature}") ~ C({group})',
        data=df,
        groups=df[sample_col]
    )

    result = model.fit()

    pval = result.pvalues[
        f"C({group})[T.2]"
    ]

    print("\n" + "=" * 70)
    print(f"FEATURE: {feature}")
    print("=" * 70)
    print(f"Mixed-effects p-value: {pval:.4g}")

    if pval < 0.0001:
        significance = "****"
    elif pval < 0.001:
        significance = "***"
    elif pval < 0.01:
        significance = "**"
    elif pval < 0.05:
        significance = "*"
    else:
        significance = "ns"

    print(f"Significance: {significance}")

    fig, ax = plt.subplots(figsize=(10, 12))

    with sns.plotting_context("notebook", font_scale=2):

        sns.violinplot(
            data=df,
            x=group,
            y=feature,
            inner="box",
            cut=0,
            scale="width",
            palette=colors,
            ax=ax
        )

        ax.set_xlabel(group, fontsize=30)
        ax.set_ylabel(feature, fontsize=30)

        ax.set_xticklabels(
            ["Sclerotic", "Non-Sclerotic"],
            fontsize=30
        )

        ax.tick_params(
            axis="y",
            labelsize=30
        )

        if "Proportion" in feature:
            ax.set_ylim(0, 1)

        ymax = df[feature].max()
        ymin = df[feature].min()
        yrange = ymax - ymin

        if yrange == 0:
            yrange = 1

        y = ymax + 0.08 * yrange

        ax.plot(
            [0, 0, 1, 1],
            [
                y,
                y + 0.02 * yrange,
                y + 0.02 * yrange,
                y
            ],
            lw=2,
            color="black"
        )

        ax.text(
            0.5,
            y + 0.03 * yrange,
            significance,
            ha="center",
            fontsize=30
        )

    plt.tight_layout()
    plt.show()
